# 🇮🇹 Italienation Open Science Observatory: Interactive Data Playground

**Esplora in autonomia i Dati Grezzi Istituzionali (`local_data`) e i Dati Elaborati (`processed_data`) / Freely explore Institutional Raw Data & Processed Panels.**

Questo notebook interattivo su **Google Colab** è progettato per consentire a ricercatori, studenti, giornalisti e cittadini di accedere istantaneamente all'intero patrimonio dati del progetto *Italienation* (685+ dataset totali), verificare le fonti istituzionali e costruire modelli analitici propri.

--- 
### 📌 Contenuto / Contents
1. **Caricamento Catalogi e Configurazione Ambiente / Environment Setup & Catalog Loading**
2. **Esplorazione Interattiva `local_data` (487 Dataset Istituzionali Grezzi / Raw Data)**
3. **Esplorazione Interattiva `processed_data` (198 Pannelli e Modelli / Processed Data)**
4. **Sandbox di Sperimentazione e Analisi Personalizzata / Custom Analysis Sandbox**

In [ ]:
# 1. CARICAMENTO CATALOGHI E SETUP AMBIENTE
# Importazione librerie essenziali
import pandas as pd
import numpy as np
import json
import urllib.request
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Base URL del repository GitHub Italienation
REPO_BASE_URL = "https://raw.githubusercontent.com/Eugenix94/Italienation/main/"

# Caricamento dei cataloghi ufficiali JSON
print("📥 Caricamento cataloghi in corso / Loading catalogs...")
raw_catalog = pd.read_json(f"{REPO_BASE_URL}web/catalog_raw.json")
proc_catalog = pd.read_json(f"{REPO_BASE_URL}web/catalog_processed.json")

print(f"✅ Catalogo Dati Grezzi (local_data): {len(raw_catalog)} dataset trovati.")
print(f"✅ Catalogo Dati Elaborati (processed_data): {len(proc_catalog)} dataset trovati.")
print(f"📊 Totale dataset pronti per l'esplorazione su Colab: {len(raw_catalog) + len(proc_catalog)}")

--- 
## 📂 2. Esplorazione e Caricamento di `local_data` (Dati Grezzi Istituzionali)

I dati grezzi (`local_data`) sono ripartiti tra le principali istituzioni italiane ed internazionali (**MIM, ISTAT, MUR, INVALSI, Eurostat, OECD, MEF/SIOPE, INPS**).
Scegli un dataset dal catalogo o usa la funzione di ricerca qui sotto!

### 🚀 Le 5 Nuove Frontiere di Espansione ed Esplorazione Scientifica
In questa sezione esploriamo interattivamente i **5 nuovi domini strategici** integrati nel catalogo via API istituzionali (`ISTAT SDMX`, `MIM SNAES`, `INDIRE/MUR`, `Infratel/AGCOM`, `legislation.gov.uk API`).

In [ ]:
# Caricamento e visualizzazione interattiva delle 5 Nuove Frontiere di Espansione
base_processed_url = "https://raw.githubusercontent.com/Eugenix94/Italienation/main/local_data/processed/"

# 1. Edilizia Scolastica e PNRR (MIM SNAES)
df_infra = pd.read_csv(base_processed_url + "school_infrastructure_seismic_safety_panel.csv")
print("=== FRONTIERA 1: EDILIZIA E SICUREZZA ANTISISMICA (MIM SNAES) ===")
display(df_infra[['Regione', 'Perc_Certificato_Agibilita', 'Perc_Verifica_Antisismica', 'PNRR_Fondi_Mense_Palestre_Mln_EUR']].head(5))

# 2. Inverno Demografico e Accorpamenti (ISTAT SDMX API)
df_demo = pd.read_csv(base_processed_url + "demographic_winter_school_closures_projection.csv")
print("\n=== FRONTIERA 2: PROIEZIONI DEMOGRAFICHE 2025-2040 (ISTAT SDMX) ===")
display(df_demo[['Regione', 'Variazione_Percentuale_2024_2035', 'Stima_Istituti_Accorpati_o_Chiusi', 'Pendolarismo_Medio_Minuti']].head(5))

# 3. ITS Academy vs Università (INDIRE / MUR)
df_its = pd.read_csv(base_processed_url + "its_academy_vs_university_outcomes.csv")
print("\n=== FRONTIERA 3: ESITI ITS ACADEMY VS LAUREE TRIENNALI (INDIRE / MUR) ===")
display(df_its[['Settore_Terziario', 'Durata_Anni', 'Tasso_Occupazione_1_Anno_Perc', 'Stipendio_Netto_Iniziale_EUR']])

# 4. Divario Digitale e Banda Ultra-Larga 1 Gbps (Infratel / AGCOM)
df_dig = pd.read_csv(base_processed_url + "digital_divide_broadband_schools_nuts3.csv")
print("\n=== FRONTIERA 4: DIVARIO DIGITALE E BANDA ULTRA-LARGA (Infratel) ===")
display(df_dig[['Regione', 'Scuole_Connesse_1Gbps_Perc', 'Laboratori_STEM_Attivi_Perc', 'Indice_DigComp_Studenti']].head(5))

# 5. Timeline Storico-Giuridica Comparata (UK vs IT)
df_leg = pd.read_csv(base_processed_url + "comparative_legal_timeline_uk_vs_italy.csv")
print("\n=== FRONTIERA 5: TIMELINE STORICO-GIURIDICA COMPARATA (legislation.gov.uk vs Normattiva) ===")
display(df_leg[['Anno', 'Paese', 'Atto_o_Riforma', 'Struttura_e_Impatto']])


In [ ]:
# Visualizza le categorie di dati grezzi disponibili
print("🏛️ Categorie di dati grezzi (local_data) per istituzione/fonte:")
print(raw_catalog['category'].value_counts())

# Funzione di ricerca rapida nei dati grezzi per parola chiave (es. 'neet', 'abbandono', 'spesa', 'invalsi')
def search_raw_datasets(keyword):
    kw = keyword.lower()
    matches = raw_catalog[
        raw_catalog['name'].str.lower().str.contains(kw) |
        raw_catalog['path'].str.lower().str.contains(kw) |
        raw_catalog['institution'].str.lower().str.contains(kw)
    ]
    return matches[['name', 'category', 'institution', 'path', 'source']]

# Esempio di ricerca: cerchiamo tutti i dataset grezzi sui NEET o abbandono scolastico
search_raw_datasets("neet").head(10)

In [ ]:
# Funzione universale per caricare qualsiasi dataset grezzo da local_data direttamente in un DataFrame
def load_raw_csv(path_or_keyword):
    if path_or_keyword.endswith(".csv"):
        target_path = path_or_keyword
    else:
        # Cerca per parola chiave e prendi il primo match
        matches = search_raw_datasets(path_or_keyword)
        if len(matches) == 0:
            raise ValueError(f"Nessun dataset trovato per la parola chiave: {path_or_keyword}")
        target_path = matches.iloc[0]['path']
        print(f"📌 Dataset selezionato: {matches.iloc[0]['name']}")
    
    full_url = f"{REPO_BASE_URL}{target_path}"
    print(f"🌐 Caricamento da GitHub: {full_url}")
    df = pd.read_csv(full_url)
    return df

# Esempio: Carichiamo un dataset ISTAT/Eurostat da local_data
df_raw_sample = load_raw_csv("openpolis_neet_metropolitan_capitals.csv")
print(f"\n📏 Dimensioni: {df_raw_sample.shape[0]} righe x {df_raw_sample.shape[1]} colonne")
df_raw_sample.head()

--- 
## ⚙️ 3. Esplorazione e Caricamento di `processed_data` (Dati Elaborati e Pannelli Analitici)

I dati elaborati (`processed_data`) contengono le serie storiche aggregate, i panel di regressione regionale, gli indici di povertà educativa e le comparazioni internazionali pronte per l'analisi statistica e l'econometria.

In [ ]:
# Funzione di ricerca e caricamento nei dati elaborati (processed_data)
def search_processed_datasets(keyword):
    kw = keyword.lower()
    matches = proc_catalog[
        proc_catalog['name'].str.lower().str.contains(kw) |
        proc_catalog['path'].str.lower().str.contains(kw)
    ]
    return matches[['name', 'category', 'path', 'source']]

def load_processed_csv(path_or_keyword):
    if path_or_keyword.endswith(".csv"):
        target_path = path_or_keyword
    else:
        matches = search_processed_datasets(path_or_keyword)
        if len(matches) == 0:
            raise ValueError(f"Nessun dataset elaborato trovato per: {path_or_keyword}")
        target_path = matches.iloc[0]['path']
        print(f"📌 Dataset elaborato selezionato: {matches.iloc[0]['name']}")
    
    full_url = f"{REPO_BASE_URL}{target_path}"
    print(f"🌐 Caricamento panel: {full_url}")
    return pd.read_csv(full_url)

# Esempio: Carichiamo il panel regionale sui NEET
df_proc_sample = load_processed_csv("neet_regional_model_panel.csv")
print(f"\n📏 Dimensioni Panel: {df_proc_sample.shape[0]} righe x {df_proc_sample.shape[1]} colonne")
df_proc_sample.head(10)

In [ ]:
# Esempio visualizzazione immediata: Tasso NEET per Macro-Area o Regione
if 'territory' in df_proc_sample.columns and 'neet_rate' in df_proc_sample.columns:
    plt.figure(figsize=(14, 6))
    sns.barplot(data=df_proc_sample.sort_values('neet_rate', ascending=False), x='territory', y='neet_rate', palette='viridis')
    plt.title('Tasso NEET Regionale (%) - Confronto dai Dati Elaborati / Regional NEET Rate (%)', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Tasso NEET 15-29 anni (%)')
    plt.xlabel('Regione / Territorio')
    plt.tight_layout()
    plt.show()

--- 
## 🛠️ 4. Sandbox di Analisi Personalizzata / Your Custom Sandbox

Usa le celle qui sotto per scrivere le tue funzioni in Python, testare correlazioni tra spesa scolastica (`siope`), bocciature (`invalsi_esiti`), redditi e abbandono, oppure esportare tabelle personalizzate in CSV!

In [ ]:
# Scrivi qui le tue analisi! Esempio:
# df_my_analysis = load_raw_csv('parole_chiave_o_nome_file.csv')
# df_my_analysis.describe()


### 📚 Approfondimento Curricolare: Comprehensive System UK vs Tripartizione Italiana
Confronto empirico sulla ripartizione oraria delle materie (STEM, Umanistiche, Economia, Design & Technology) e sull'Indice di Segregazione del Capitale Culturale (`Habitus di Bourdieu`).

In [ ]:
# Caricamento e analisi del quadro disciplinare comparato (Fascia 14-16 e Superiore)
base_url = "https://raw.githubusercontent.com/Eugenix94/Italienation/main/local_data/processed/"
df_curr = pd.read_csv(base_url + "curriculum_subjects_tripartite_vs_comprehensive_panel.csv")
print("=== CONFRONTO CURRICOLARE: COMPREHENSIVE UK VS TRIPARTIZIONE IT ===")
display(df_curr[['Paese', 'Sistema_o_Indirizzo', 'Età_Studenti', 'Totale_Materie_Distinte_Insegnate', 'Indice_Poliedricita_Curricolare_0_10', 'Segregazione_Capitale_Culturale_Score_0_10']])